# Лабораторная работа №9  
# Оценка качества RAG-систем

---

**Выполнил(а):** студент(ка) группы ______  
**ФИО:** _______________________________  
**Дата:** _______________________________

---

## 1. Цель работы

- Изучить метрики оценки качества RAG-систем
- Освоить фреймворк RAGAS для автоматической оценки
- Научиться оценивать релевантность и точность ответов
- Сравнить различные подходы к retrieval
- Проанализировать влияние качества документов на ответы

---

## 2. Теоретические сведения

### 2.1. Метрики оценки RAG

| Метрика | Что измеряет | Диапазон |
|---------|--------------|----------|
| Faithfulness | Соответствие ответа контексту | 0-1 |
| Answer Relevance | Релевантность ответа вопросу | 0-1 |
| Context Precision | Точность извлечения контекста | 0-1 |
| Context Recall | Полнота извлечения контекста | 0-1 |

### 2.2. Фреймворк RAGAS

**RAGAS** (Retrieval Augmented Generation Assessment) — фреймворк для оценки RAG без эталонных ответов:
- Использует LLM для оценки качества
- Поддерживает multiple metrics
- Работает с любыми RAG пайплайнами

### 2.3. Критерии качества

- **Ответ основан на контексте** (не выдуман)
- **Полнота ответа** (все аспекты вопроса раскрыты)
- **Релевантность retrieved документов**

---

## 3. Задание

### 3.1. Базовый уровень (обязательно)

1. Установите и импортируйте RAGAS
2. Создайте тестовый датасет с вопросами и ответами
3. Оцените качество по метрикам Faithfulness и Answer Relevance
4. Проанализируйте результаты
5. Сделайте выводы о качестве системы

### 3.2. Продвинутый уровень (дополнительно)

- Добавьте метрики Context Precision и Context Recall
- Сравните два разных подхода к retrieval
- Визуализируйте распределение метрик

### 3.3. Индивидуальное задание

Проведите оценку для вашей RAG системы из ЛР3:
- **Вариант A:** Оценка на синтетических вопросах
- **Вариант B:** Оценка на реальных вопросах пользователей

Предложите пути улучшения качества.

---

## 4. Ход работы

### 4.1. Подготовка окружения

In [ ]:
# Установка зависимостей
!pip install ragas datasets -q
!pip install langchain langchain-openai -q

# Проверка установки
import ragas
print(f"RAGAS version: {ragas.__version__}")

### 4.2. Импорт библиотек и настройка

In [ ]:
import os
from google.colab import userdata

# Настройка API ключа
try:
    os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
except:
    os.environ["OPENAI_API_KEY"] = "your-api-key-here"

from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall

print("Библиотеки импортированы!")

### 4.3. Создание тестового датасета

In [ ]:
# Пример данных для оценки
data_samples = {
    'question': [
        'Какие преимущества у квантования моделей?',
        'Что такое LoRA и как она работает?',
        'Как работает механизм внимания в Transformer?',
        'Какие бывают типы токенизации?'
    ],
    'answer': [
        'Квантование уменьшает размер модели и ускоряет инференс за счет снижения битности весов.',
        'LoRA (Low-Rank Adaptation) добавляет обучаемые низкоранговые матрицы для эффективного fine-tuning.',
        'Механизм внимания вычисляет веса важности между токенами для учета контекстных зависимостей.',
        'Основные типы: WordPiece, BPE, SentencePiece, Unigram.'
    ],
    'contexts': [
        ['Квантование снижает точность представления весов модели, что уменьшает потребление памяти и ускоряет вычисления.'],
        ['LoRA замораживает основные веса модели и обучает только небольшие адаптерные слои с низкой ранговой декомпозицией.'],
        ['Attention механизм позволяет модели фокусироваться на релевантных частях входной последовательности при генерации каждого токена.'],
        ['Токенизация преобразует текст в числовые ID. BPE разбивает слова на подслова, WordPiece использует максимальное совпадение.']
    ],
    'ground_truth': [
        'Квантование уменьшает размер модели, снижает требования к памяти и ускоряет инференс.',
        'LoRA это метод эффективного дообучения через низкоранговые матрицы.',
        'Attention вычисляет взвешенную сумму значений на основе схожести запроса и ключей.',
        'BPE, WordPiece, SentencePiece, Unigram токенизация.'
    ]
}

# Создание Dataset
dataset = Dataset.from_dict(data_samples)
print(f"Датасет создан: {len(dataset)} примеров")
print(dataset[0])

### 4.4. Оценка качества

In [ ]:
# Выбор метрик для оценки
metrics = [
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall
]

print("Начало оценки...")
result = evaluate(
    dataset,
    metrics=metrics,
    raise_exceptions=False
)

print("\n=== Результаты оценки ===")
print(result)

### 4.5. Анализ результатов

In [ ]:
import pandas as pd

# Конвертация в DataFrame для анализа
df = result.to_pandas()
print("\nДетальные результаты:")
print(df[['question', 'faithfulness', 'answer_relevancy', 'context_precision', 'context_recall']])

# Средние значения
print("\n=== Средние значения метрик ===")
for col in ['faithfulness', 'answer_relevancy', 'context_precision', 'context_recall']:
    if col in df.columns:
        mean_val = df[col].mean()
        print(f"{col}: {mean_val:.3f}")

### 4.6. Визуализация результатов

In [ ]:
import matplotlib.pyplot as plt

# Построение графика
metrics_names = ['Faithfulness', 'Answer Relevancy', 'Context Precision', 'Context Recall']
metrics_values = [
    df['faithfulness'].mean() if 'faithfulness' in df.columns else 0,
    df['answer_relevancy'].mean() if 'answer_relevancy' in df.columns else 0,
    df['context_precision'].mean() if 'context_precision' in df.columns else 0,
    df['context_recall'].mean() if 'context_recall' in df.columns else 0
]

plt.figure(figsize=(10, 6))
plt.bar(metrics_names, metrics_values, color=['#2ecc71', '#3498db', '#e74c3c', '#f39c12'])
plt.ylim(0, 1)
plt.title('Метрики качества RAG-системы')
plt.ylabel('Score (0-1)')
plt.axhline(y=0.7, color='r', linestyle='--', alpha=0.5, label='Порог качества')
plt.legend()
plt.tight_layout()
plt.show()

---

## 5. Контрольные вопросы

1. Какие метрики используются для оценки RAG?
2. В чем разница между Faithfulness и Answer Relevance?
3. Почему важно оценивать качество retrieved контекста?
4. Как интерпретировать значения метрик?
5. Какие пороги значений считаются приемлемыми?

---

## 6. Выводы

В ходе работы были изучены метрики оценки RAG-систем, проведена оценка с помощью фреймворка RAGAS и проанализированы результаты.

---